### Nama : Defirdas A.S.
### NIM : 3222684

## DETEKSI PLAT NOMOR DENGAN METODE HAARCASCADE DAN OCR

#### Langkah - langkah :

1. Input foto berupa kendaraan
2. Mendeteksi plat
3. Mengidentifikasi nomor dan huruf pada plat
4. Dapatkan identitas plat nomor

In [1]:
import cv2
import numpy as np
import pytesseract

Objek cascade

In [2]:
cascade = cv2.CascadeClassifier("haarcascade_russian_plate_number.xml")

In [3]:
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

Plat nomor wilayah Indonesia

In [17]:
daftar_kode_wiLayah = {
            "B":"DKI Jakarta, Depok, Bekasi, Tangerang", 
            "D":"Bandung, Bandung Barat, Cimahi", 
            "E":"Cirebon, Indramayu, Majalengka, Kuningan", 
            "F":"Bogor, Sukabumi", 
            "T":"Karawang, Purwakarta, Subang", 
            "A":"Banten", 
            "G":"Pekalongan, Batang, Pemalang, Tegal", 
            "H":"Semarang", 
            "K":"Pati, Kudus, Jepara, Rembang, Blora, Grobogan", 
            "R":"Banyumas, Banjarnegara, Cilacap, Purbalingga", 
            "AA":"Kedu, Magelang, Temanggung, Wonosobo, Purworejo, Kebumen", 
            "AD":"Surakarta, Sukoharjo, Klaten, Karanganyar, Sragen, Wonogiri, Boyolali", 
            "AB":"Yogyakarta, Sleman, Bantul, Gunungkidul, Kulon Progo", 
            "BA":"Sumatera Barat", 
            "BB":"Sumatera Utara bagian Barat", 
            "BD":"Bengkulu", 
            "BE":"Lampung", 
            "BG":"Sumatera Selatan", 
            "BH":"Jambi", 
            "BK":"Sumatera Utara bagian Timur", 
            "BL":"Aceh", 
            "BM":"Riau", 
            "BN":"Bangka Belitung", 
            "BP":"Kepulauan Riau", 
            "KB":"Kalimantan Barat", 
            "DA":"Kalimantan Selatan", 
            "KH":"Kalimantan Tengah", 
            "KT":"Kalimantan Timur", 
            "KU":"Kalimantan Utara", 
            "DB":"Sulawesi Utara", 
            "DC":"Sulawesi Barat", 
            "DD":"Sulawesi Selatan", 
            "DE":"Sulawesi Tenggara", 
            "DN":"Sulawesi Tengah", 
            "DM":"Gorontalo", 
            "DT":"Sulawesi Tenggara", 
            "DK":"Bali", 
            "DR":"Nusa Tenggara Barat", 
            "EA":"Nusa Tenggara Timur", 
            "DE":"Maluku", 
            "DG":"Maluku Utara", 
            "DS":"Papua Barat", 
            "DX":"Papua",

}

### Mendeteksi Plat

In [5]:
def baca_kode(string):
    if not string:
        return f"Masalah deteksi plat, {string}"

    prefix = string[0:2]
    kode_prefix = ''
    for char in prefix:
        if char.isalpha():
            kode_prefix += char
        else:
            break

    if kode_prefix in daftar_kode_wilayah:
        wilayah = daftar_kode_wilayah[kode_prefix]
        return f"Kendaraan wilayah {wilayah}."
    else:
        return 'Kode wilayah tidak dikenali!'
    

In [24]:
def deteksi_nomor(gambar):
    global read
    img = cv2.imread(gambar)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    plat = cascade.detectMultiScale(gray, 1.1, 4) 

    # looping bila deteksi lebih dari satu
    for (x, y, w, h) in plat:
        # mengambil region of interest (ROI) objek
        # menentukan space 20% dari ukuran gambar
        # a,b = (int(0.02*img.shape[0]), int(0.02*img.shape[1])) # a = height, b = width

        # padding 10%
        a,b = (int(0.01*img.shape[0]), int(0.01*img.shape[1])) # a = height, b = width

        #mengambil area di dalam padding
        plat_nomor = img[y+a:y+h-a, x+b:x+w-b, :]


        ## IMAGE PROCESSING
        
        kernel = np.ones((1,1), np.uint8)
        
        # melakukan dilasi untuk menebalkan karakter
        plat_nomor = cv2.dilate(plat_nomor, kernel, iterations=1)

        #melakukan erosi untuk menghilangkan noise
        plat_nomor = cv2.erode(plat_nomor, kernel, iterations=1)

        # mengubah gambar plat ke grayscale
        plat_gray = cv2.cvtColor(plat_nomor, cv2.COLOR_BGR2GRAY)

        # menentukan threshold untuk mengubah gambar menjadi warna biner
        (threshold, plat_nomor) = cv2.threshold(plat_gray, 127, 255, cv2.THRESH_BINARY)

        
        ## IDENTIFIKASI NOMOR PLAT

        read = pytesseract.image_to_string(plat_nomor)
        print(read)

        if read is None:
            print('Gagal membaca plat nomor!')
            return

        else:
            read = ''.join(e for e in read if e.isalnum())
            wilayah = baca_kode(read)
            print(wilayah)

            ## BOUNDING BOX
            
            cv2.rectangle(img, (x,y), (x+w, y+h), (51,51,255), 2)
            cv2.rectangle(img, (x, y-40), (x+w, y), (51,51,255), -1)
            cv2.putText(img, read, (x,y-10), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 1, cv2.LINE_AA)
            cv2.imshow('Plat', plat_nomor)
        
    cv2.imshow('Hasil', img)
    cv2.imwrite('hasil.jpg', img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

In [25]:
deteksi_nomor('img/img2.jpg')

00-1247 St



NameError: name 'daftar_kode_wilayah' is not defined